# 01 · Structure and completeness

**PRT2030 · Dataset inspection · Study notebook**

Before comparing causes, years or fire sizes, establish what the file contains and what information is unavailable.
We will move from the whole table to individual missing values, overlapping absences and repeated records.
Finally, we will turn the missing-containment finding into evidence and one complete issue-log entry.

**How to use this study resource:** Read the inspection question, follow the comments as you run the code,
and explain what the output tells you. Run from the top; later checks reuse named results from earlier cells.
Each notebook loads its own data and runs independently. Use your existing `wildfires.csv` beside the notebooks.

**One source row represents a reported wildfire record.** Matching names or rows do not establish that records
describe the same physical fire: this extract has no incident identifier. Fire sizes are in acres.
We inspect temporary results and evidence tables while keeping `raw` unchanged.

The worked findings refer to the supplied teaching extract (1,880,465 records and eight columns).
They are reference figures from the supplied notebooks, not results of a new run in this revision.
Code outputs have been cleared so your next top-to-bottom run produces a consistent set of results.
If your file differs, use your own computed results. The teaching target remains pandas 3.0.5.
The worked log wording describes the reference findings; revise the written judgements if your results differ.

Function and method names in the documentation lines are clickable links to their specific official reference pages.

**Reading the examples:** The small values introduced as examples explain the logic;
they are not new measurements from the wildfire file. Comments explain the purpose,
intermediate results and reasons for the choices. Apply the stated prerequisites
before interpreting a zero flag count as a pass.


## 1. Load the data and recognise the objects

| Kind of object | Meaning in these examples |
| --- | --- |
| DataFrame | A table with rows and columns; a result table may have a different row meaning from the source. |
| Series | One labelled sequence of values, such as one name per record or one count per column. |
| Boolean Series | One True/False decision per labelled item; often used to select source rows. |
| Single value | One count, percentage or boundary, rather than a whole column. |

The index contains row labels. Results derived from the same source keep those labels so values can be matched correctly.
Start by loading the original table and establishing the whole-file record count.

**Documentation:** [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len) · [print()](https://docs.python.org/3.14/library/functions.html#print) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html).


In [ ]:
# We need a trustworthy starting table before we can judge its values. Keep this import as
# raw so every finding can be traced back to the same original records.

# Pandas gives us labelled tables and columns. The short name pd lets us use that library
# consistently in all later cells.
import pandas as pd

# The CSV becomes a DataFrame: one row is one reported fire, and each column supplies a
# different fact about it. Loading it afresh lets this notebook run independently of the
# others.
raw = pd.read_csv("wildfires.csv")

# A missing-value check asks about records, so its whole-file denominator must be the row
# count. Counting all cells would make the affected percentage artificially small.
total_records = len(raw)
# Stop on an empty import: there is no population to inspect and division by the row count
# would be undefined. An empty file cannot demonstrate that its values passed a check.
if total_records == 0:
    raise ValueError("The dataset contains no records. Check the import before continuing.")

# Record the environment alongside the count so an unexpected result can be compared with
# another run. A mixed-type warning is a clue to inspect a column; it does not identify which
# values are wrong.
print("pandas version:", pd.__version__)
print("Reported wildfire records:", total_records)


## 2. Understand the table before judging its values

The import has produced a table. First establish whether its layout and fields match the extract we intended to load;
then inspect a few records and how pandas stores each column.


### 2.1 Dimensions and column names

An unexpected number of columns or unfamiliar headings can indicate an import or file-selection problem. These basic checks establish the structure for everything that follows.

**Inspection question:** What table did we load?

**Documentation:** [DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html) · [DataFrame.size](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.size.html) · [DataFrame.columns](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) · [DataFrame.index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.index.html) · [Index.tolist()](https://pandas.pydata.org/docs/reference/api/pandas.Index.tolist.html).


In [ ]:
# Before inspecting values, check that the file was separated into the rows and columns we
# expected.

# Read shape as (records, fields). For example, (100, 8) means 100 records with eight fields;
# (100, 1) could suggest that the delimiter was interpreted incorrectly.
print("Rows and columns:", raw.shape)

# With 100 rows and eight columns, size is 800. This includes missing cells, so size is
# neither the number of fires nor the number of available values.
print("Total cells:", raw.size)

# The labels determine which fields later expressions select. An unexpected heading can
# explain a failed selection even when the table dimensions look correct.
print("Column names:", raw.columns.tolist())

# The default labels 0, 1, 2, ... let us match later flags and evidence to the imported rows.
# A row label identifies a position in this import, not a physical fire.
print("Row labels:", raw.index)


### 2.2 Preview the beginning and end

The dimensions tell us how much data exists, but not what its values look like. Inspect both ends for obvious gaps, formats and ordering.

**Inspection question:** Do these records look like the information we expected?

**Documentation:** [DataFrame.head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) · [DataFrame.tail()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.tail.html) · [display()](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.display).


In [ ]:
# A few actual records help us check whether the import looks plausible before we calculate
# summaries.

# Compare the beginning and end: a file sorted by year may show different periods at each
# end. Explicit display calls show both tables; the notebook otherwise displays only the
# final bare expression.
display(raw.head(5))
# Look for unexpected blank fields, shifted values or date formats. These ten displayed
# records suggest what to investigate; they cannot tell us how common a problem is.
display(raw.tail(5))


### 2.3 Look elsewhere in the table

The first and last records may share an order or source. A sample from across the table can reveal other formats worth checking.

**Inspection question:** What variation can we see away from the ends of the file?

**Documentation:** [DataFrame.sample()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) · [min()](https://docs.python.org/3.14/library/functions.html#min).


In [ ]:
# Inspecting only the ends can miss other formats or periods, so take a small sample from
# across the imported table.

# If the file has only three records, requesting five without replacement would fail. Taking
# the smaller of five and the actual row count keeps the preview valid for small files.
sample_size = min(5, total_records)

# Using the same random seed makes it easier to discuss the same examples on repeated runs of
# the same input. Five sampled rows still cannot establish that the rest of the file is
# valid.
display(raw.sample(n=sample_size, random_state=42))


### 2.4 Compare with the expected structure — S01

A **schema** describes the structure we expect. Use the file documentation to set the expectation.

| Field | Meaning | Expected on CSV import |
| --- | --- | --- |
| `fire_name` | Reported name | Text, possibly missing |
| `state` | State or jurisdiction code | Text |
| `discovery_date` | When the fire was discovered | Text to be tested as a date |
| `contained_date` | When the fire was contained | Text to be tested as a date, possibly missing |
| `fire_year` | Reported fire year | Whole number |
| `fire_size` | Area in acres | Number |
| `fire_class` | Size class A–G | Text |
| `fire_cause` | Reported cause | Text |

**Documentation:** [set()](https://docs.python.org/3.14/library/stdtypes.html#set).

**Structure rule example:** Suppose the file contains all expected headings except `contained_date`,
and adds `county`. The missing-field result should contain `contained_date`; the extra-field result
should contain `county`. The missing endpoint prevents the later date checks from running, while
the extra location field may be useful. Changing only column order does not violate this heading-presence rule.


In [ ]:
# A structure check needs an expectation set before inspecting the result; otherwise we would
# accept whatever headings happened to arrive.

# These eight documented fields support our planned questions. For example, without
# contained_date we cannot even run the duration-availability check.
expected_columns = [
    "fire_name", "state", "discovery_date", "contained_date",
    "fire_year", "fire_size", "fire_class", "fire_cause"
]

# Expected minus observed keeps required labels that did not arrive. Observed minus expected
# does the reverse: it finds additions. Sets ignore order because a different column order
# does not make a required field absent.
missing_columns = set(expected_columns) - set(raw.columns)
extra_columns = set(raw.columns) - set(expected_columns)
print("Missing expected columns:", missing_columns)
print("Unexpected columns:", extra_columns)

# A non-empty set means a prerequisite is missing. Stop here so later errors do not disguise
# the real import problem. Extra columns are reported for review because they may contain
# useful additional information.
if missing_columns:
    raise ValueError("Required fields are missing. Resolve the import before continuing.")


### 2.5 Storage types and completeness

The required headings are present. Next compare how their values are stored with what the fields mean: a date-looking string is still text until interpreted as a date.

**Inspection question:** Which storage types or incomplete columns need further investigation?

**Documentation:** [DataFrame.dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) · [DataFrame.info()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html).


In [ ]:
# A column heading tells us its intended meaning; its storage type tells us which operations
# Pandas can currently perform.

# A date stored as text cannot yet be subtracted as a date. A numeric storage type permits
# arithmetic but does not prove that a size is positive or a year is sensible.
display(raw.dtypes)

# On a large table, explicitly request non-missing counts so incomplete fields remain
# visible. Deep memory reporting includes more of the text storage, which helps explain why
# large copies can consume substantial memory.
raw.info(show_counts=True, memory_usage="deep")


### 2.6 Check the original CSV header

pandas can rename repeated or empty headings during import. Read the source header directly to check labels that the imported table might conceal.

**Inspection question:** Are any source headings repeated, blank or padded with whitespace?

**Documentation:** [open()](https://docs.python.org/3.14/library/functions.html#open) · [next()](https://docs.python.org/3.14/library/functions.html#next) · [csv.reader()](https://docs.python.org/3.14/library/csv.html#csv.reader) · [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [Series.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html) · [Series.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html).


In [ ]:
# Inspect the CSV header itself because an importer may rename duplicate or blank labels,
# hiding the original heading problem.

import csv
# Read the first CSV record with a CSV reader rather than splitting on commas: quoted fields
# may contain commas. The encoding handles an optional leading marker; the with block closes
# the file even if reading fails.
with open("wildfires.csv", encoding="utf-8-sig", newline="") as source:
    header = next(csv.reader(source))

# Here one Series item means one heading position, not one fire. Preserve that position so
# two identical heading labels can still be located separately.
header_names = pd.Series(header, dtype="string")
display(header_names)

# If the header contains state twice, both positions need review. keep=False marks every
# occurrence, whereas the default would leave the first one unflagged.
repeated_header = header_names.duplicated(keep=False)

# A heading containing only spaces looks populated in the file but gives no usable label.
# Trimming for comparison makes it empty; comparing original and trimmed labels separately
# reveals padded labels such as " state ".
blank_header = header_names.str.strip() == ""
spaced_header = header_names != header_names.str.strip()

# Show labels for spelling problems and positions for blank labels: an empty string alone
# would not tell the reviewer which column to find. Empty results mean only that these
# particular screens found nothing.
print("Repeated labels:", header_names.loc[repeated_header].tolist())
print("Blank positions:", header_names.loc[blank_header].index.tolist())
print("Padded labels:", header_names.loc[spaced_header].tolist())


## 3. Summarise values and distinct choices

The structure is now understood. Summaries help us decide which values deserve closer inspection;
they are starting evidence, not automatic error reports.


### 3.1 Numerical summary

Fire size may range from tiny incidents to very large fires. Compare typical sizes with the spread and largest values before defining a review rule.

**Inspection question:** How are the recorded fire sizes distributed?

**Documentation:** [Series.describe()](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html).

**What would a size-review rule look like?** One later screen flags a positive finite size when it
falls below the lower IQR fence or above the upper fence. For **illustrative** fences of -5 and
11 acres, a 12-acre measurement is flagged and an 11-acre measurement is not. These are computed
review boundaries, not a universal maximum valid fire size. This summary helps us understand the
distribution before applying that screen in Notebook 03.


In [ ]:
# We need to understand typical fire sizes and their spread before deciding what counts as an
# unusual measurement.

# Summarising just fire_size keeps all numerical results in acres, apart from the count of
# available measurements. If sizes were stored as text, this would be a text summary instead;
# check the type above.
size_summary = raw["fire_size"].describe()

# Compare the mean with the 50% value (median): a few very large fires can raise the mean
# while the median still describes a small typical fire. The 25% and 75% values bound the
# middle half; std describes spread in acres, not a percentage.
# The minimum and maximum tell us where to look next. A huge maximum might be a real fire, so
# this summary supplies an investigation question rather than a correction.
display(size_summary)


### 3.2 Summary of all columns

Numerical summaries describe measurements. Text summaries instead reveal distinct labels and frequently repeated values.

**Inspection question:** What do numerical and text summaries tell us about the different fields?

**Documentation:** [DataFrame.describe()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html).


In [ ]:
# Different fields need different summaries: measurements have averages, while labels have
# frequencies and distinct choices.

# include="all" avoids silently leaving text fields out of the inspection. The summary table
# has statistic names down its rows and source fields across its columns.
all_summaries = raw.describe(include="all")

# For text, top is a most frequent value and freq is its occurrence count: top="CA", freq=300
# would mean 300 records contain CA. A mean under a text field is unavailable because
# averaging labels has no meaning; that summary NaN is not a missing source record.
display(all_summaries)


### 3.3 Constant and entirely missing columns

A field with no observed values provides no information. A field with one observed value cannot distinguish records by that value, though it may still contain missing cells.

**Inspection question:** Does every column contribute some variation?

**Documentation:** [DataFrame.nunique()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html) · [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html) · [Index.tolist()](https://pandas.pydata.org/docs/reference/api/pandas.Index.tolist.html).


In [ ]:
# Check whether a field supplies any observed information and whether that information varies
# enough to distinguish records.

# Missing cells are excluded: [CA, CA, missing] has one observed distinct value; [missing,
# missing] has zero. That difference separates limited variation from complete absence.
distinct_counts = raw.nunique()
display(distinct_counts)

# Filtering this Series selects counts, but its index contains the field names we need to
# report. A zero count identifies a field with no observed values, not a fire with missing
# information.
empty_columns = distinct_counts.loc[distinct_counts == 0].index.tolist()
# One observed value means the field cannot split its observed records into different value
# groups. It may still be useful context, and missing records may still exist, so neither
# result alone justifies dropping a field.
constant_columns = distinct_counts.loc[distinct_counts == 1].index.tolist()
print("Entirely missing columns:", empty_columns)
print("One observed value:", constant_columns)


## 4. Find and count missing information

We now move from summaries to a specific limitation: unavailable information. Begin with one field, then
extend the same operation to all fields and examine how their missing values overlap.


### 4.1 Start with one column — M01

A record without a name cannot support a name-based investigation in the same way as a named record. Its size, state or cause may still be useful.

**Inspection question:** How many records have no fire name?

**Documentation:** [Series.isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame.head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).

**Absence example:** A missing name is flagged by this check. The text `unknown` is a present value,
so it is not flagged here even if it gives us little useful information. A separate placeholder
screen in Notebook 02 investigates that meaning.


In [ ]:
# To measure missing names, first make one decision for every source record, then count the
# decisions that answer yes.

# A Boolean Series stores True where the name is absent and False where it is present.
# Keeping the original row labels lets the same decisions later select the correct source
# records.
missing_names = raw["fire_name"].isna()

# Summing flags counts affected records because True contributes 1 and False contributes 0.
# For [True, False, True], the sum is 2; this measures the full column, not a displayed
# sample.
missing_name_count = missing_names.sum()
print("Records without a fire name:", missing_name_count)

# The flags tell loc which rows to keep; head limits only the display. These records may
# still have useful sizes or causes. Text such as "unknown" is present, so it needs a
# different check from isna.
display(raw.loc[missing_names].head(10))


### 4.2 Put the count into context

The affected count tells us how many names are absent. A percentage tells us how much of the complete dataset is affected.

**Inspection question:** What proportion of all records lacks a name?

**Documentation:** [round()](https://docs.python.org/3.14/library/functions.html#round).


In [ ]:
# The count needs a denominator to show how much of the inspected population lacks a name.

# If 20 of 100 records lack names, 20 / 100 is 0.2 of the population; multiplying by 100
# gives 20%. Use records because each flag refers to a record, not to all eight cells in that
# record.
missing_name_percent = missing_name_count / total_records * 100

# Rounding makes the output easier to read. It changes the displayed precision, not the
# affected count or the source data.
print("Missing names (% of all records):", round(missing_name_percent, 2))


### 4.3 Repeat the check across the table

Names are only one field. Missing containment dates could prevent duration calculations, so we need a completeness report covering every column. Each report row will represent a field, not a fire.

**Inspection question:** How many values are missing in each column, and what percentage of records does that represent?

**Documentation:** [DataFrame.isna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) · [DataFrame.sum()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) · [DataFrame.notna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [Series.round()](https://pandas.pydata.org/docs/reference/api/pandas.Series.round.html).


In [ ]:
# Build one completeness report so we can compare information availability across fields
# using the same whole-file population.

# The Boolean table has the same rows and columns as raw. A True at row 17, contained_date
# means that specific record lacks that specific field.
missing_flags = raw.isna()

# axis=0 collapses the source rows by adding down each column. The result now has one item
# per field: its labels are fire_name, state, ... rather than source-row numbers.
missing_counts = missing_flags.sum(axis=0)

# Each field had one opportunity to provide a value in every record. Dividing each missing
# count by total_records therefore makes the fields comparable.
missing_percentages = missing_counts / total_records * 100

# Count available values as a cross-check: for each field, present plus missing should equal
# total_records. This also identifies the population available for later checks requiring a
# present value.
present_counts = raw.notna().sum(axis=0)

# Change the meaning of a row deliberately: this new DataFrame has one row per FIELD.
# Assigning the first Series supplies those field-name row labels.
missing_report = pd.DataFrame()
missing_report["Missing rows"] = missing_counts

# Pandas matches subsequent Series by label, so a fire_name percentage belongs beside the
# fire_name count even if another Series is ordered differently. Alignment would fail if we
# mixed field labels with source-row labels.
missing_report["Missing percent"] = missing_percentages.round(2)
missing_report["Present rows"] = present_counts
display(missing_report)


### 4.4 Look across each row

Adding the missing counts for all columns can count the same record several times. We therefore change direction and ask about each record as a whole.

**Inspection question:** How many records have at least one missing field, and how many fields are missing per record?

**Documentation:** [DataFrame.any()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html) · [DataFrame.all()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.all.html) · [DataFrame.sum()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) · [Series.value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [Series.sort_index()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_index.html).


In [ ]:
# A record missing two fields must count once when we ask whether any information is absent,
# but twice when we count its missing fields.

# axis=1 works across each source row. For [False, True, True], any gives True because at
# least one field is missing; summing those row flags later counts that record once.
rows_with_missing = missing_flags.any(axis=1)
# all requires every field flag to be True. A record missing a name but containing a size
# fails this whole-row-absence condition because it still supplies some information.
entirely_missing = missing_flags.all(axis=1)
# For the same [False, True, True] example, sum gives 2. This Series measures missing FIELDS
# within each record, which is a different question from how many records are affected.
missing_per_row = missing_flags.sum(axis=1)
print("Rows with any missing field:", rows_with_missing.sum())
print("Rows with every field missing:", entirely_missing.sum())

print("Missing fields per record -> number of records:")
# Group records by that missing-field count. A result labelled 2 with value 50 means 50
# records each lack two fields; sorting the labels makes 0, 1, 2, ... easy to follow.
display(missing_per_row.value_counts().sort_index())


### 4.5 Understand overlapping missing values

Some records can lack both a name and a containment date. We need the four combinations of present/missing values to see this overlap clearly. Read each output cell using both its row label and its column label.

**Inspection question:** Do absent names and absent containment dates occur in the same records?

**Documentation:** [pd.crosstab()](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [DataFrame.reindex()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html).


In [ ]:
# We need to see whether the two missing-field populations overlap before combining their
# counts.

# Use the same imported row labels as missing_names so the name and date decision refer to
# the same fire record.
missing_containment = raw["contained_date"].isna()

# Each record enters exactly one of four groups. For example, name=False and containment=True
# means a name is available but the containment date is absent.
overlap = pd.crosstab(missing_names, missing_containment)

# An unobserved category can disappear from a cross-tabulation. Add both Boolean categories
# in a known order and show absent groups as zero so all four possibilities remain readable.
overlap = overlap.reindex(index=[False, True], columns=[False, True], fill_value=0)

# Translate the flag meaning in the labels: False in a missing-value flag means PRESENT. The
# top-right cell therefore counts named records without a containment date.
overlap.index = ["Name present", "Name missing"]
overlap.columns = ["Containment present", "Containment missing"]
# The bottom-right group belongs to both missing populations. Adding the two field counts
# counts it twice, so subtract that overlap once: 960,479 + 891,531 - 651,288 = 1,200,722 in
# the reference extract.
display(overlap)


## 5. Investigate repeated records — R01

Completeness checks locate absent information. We now ask whether the same recorded values occur more than once.
Two duplicate-counting settings answer different questions, so practise on four values before checking the full table.


### 5.1 Understand the two duplicate counts

This small demonstration is separate from the wildfire data. Three occurrences of A form one repeated group: two are later copies, but all three belong to the group.

**Inspection question:** Why do the default duplicate count and keep=False give different totals?

**Documentation:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [Series.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).


In [ ]:
# Use four values to separate two questions: how many later copies exist, and how many items
# belong to repeated groups?

# Three A values form one repeated group. B occurs only once and should stay outside that
# group.
example = pd.Series(["A", "A", "A", "B"])

# Put values and decisions side by side so we can explain the counts rather than memorise
# them. The default marks A occurrences two and three: two later copies. keep=False also
# marks the first A: three group members.
example_report = pd.DataFrame({
    "Value": example,
    "Later repetition": example.duplicated(),
    "All repeated members": example.duplicated(keep=False)
})
# The difference of one represents the first member of this repeated group. Neither setting
# decides whether repeated values describe the same real-world event.
display(example_report)


### 5.2 Apply the check to complete wildfire rows

We now compare all eight fields together. Matching rows are candidates for investigation; without an incident identifier, they are not confirmed duplicate fires.

**Inspection question:** How many complete rows repeat, and what do those records contain?

**Documentation:** [DataFrame.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [DataFrame.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html).


In [ ]:
# Find records whose complete stored values match, then inspect their context before deciding
# what the repetitions mean.

# Compare every source column, not only the fire name: two different fires can share a name.
# The DataFrame index is not part of this value comparison.
later_repetitions = raw.duplicated()
# Keep every matching partner as a review candidate. For a group of three identical rows, the
# default counts two later copies and keep=False counts all three.
repeated_members = raw.duplicated(keep=False)
print("Later repetitions:", later_repetitions.sum())
print("All repeated-group members:", repeated_members.sum())

# Every repeated group has exactly one first occurrence omitted by the default. Subtracting
# later copies from all members therefore counts repeated groups, not unique physical fires.
print("Repeated groups:", repeated_members.sum() - later_repetitions.sum())

# Select full records using the membership flags, then sort by all expected fields to put
# identical partners together. An unsorted ten-row preview might show one partner without the
# others.
repeated_rows = raw.loc[repeated_members]

repeated_rows = repeated_rows.sort_values(expected_columns)
# The full repeated_rows table keeps all flagged records; head changes the display only.
# Matching values establish repeated recorded rows, while incident identity still requires
# source information.
display(repeated_rows.head(10))


## 6. Worked log entry: missing containment dates — M02

The completeness report identified absent containment dates. We now follow that finding from detection to evidence
and a justified issue-log entry. Our question is: **Can we calculate discovery-to-containment duration for every record?**
Both dates are needed. Missing containment dates prevent that calculation, even if the original reporting system permits them.
For this all-record duration question, the worked severity is **High**; a cause-based question may be affected much less.

| Object we will build | What it contains | Meaning of one row |
| --- | --- | --- |
| `evidence_m02` | Original values supporting M02, plus reference labels | One affected source record |
| `log_entry` | A Python dictionary containing the answers for one issue | Not a table yet: field names paired with answers |
| `issue_log` | The dictionary converted into a DataFrame | One issue, however many records support it |

`source_row` preserves the original zero-based pandas row label from this unchanged import. It is not an incident ID
or a physical CSV line number. `issue_id` connects the evidence to its issue-log entry.

Some log fields are measured by code; others require your judgement. State the question before choosing severity.

| Severity | Meaning for the stated analysis |
| --- | --- |
| Low | Limited effect on the stated question. |
| Medium | A meaningful part of the comparison or grouping may be affected. |
| High | A central result cannot be supported reliably without a decision. |
| Not assigned | More investigation is needed before judging the effect. |

A percentage alone does not select a severity. A **proposed action** is a plan; **status** records progress;
**verification** explains how the later decision or treatment will be checked.


### 6.1 Identify and count the affected records

Repeat the missing-date rule here so the complete evidence-to-log process is visible in one section.

**Inspection question:** Which records cannot provide a containment date for a duration calculation?

**Documentation:** [Series.isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html).

**Missing-date rule:** Flag a record when its original containment value is absent. A record with
discovery on 10 July and no containment date is flagged because duration has no second endpoint.
A record with both dates present is not flagged by this absence check; those dates still need
conversion and order checks. Every source record can be checked for absence, so there is no
presence prerequisite for M02.


In [ ]:
# Recreate the M02 finding here so the evidence and log are built from the exact decisions we
# count.

# Every record can be checked for absence, even when the discovery date is also missing. True
# means this record cannot supply its containment endpoint for a duration calculation.
missing_containment = raw["contained_date"].isna()
# Count all True flags before making a preview. If 500 records match but we display ten, the
# issue count must remain 500.
missing_containment_count = missing_containment.sum()
print("Missing containment dates:", missing_containment_count)
print("Records checked:", total_records)


### 6.2 Build and inspect the full evidence table

A reviewer needs the original absent value, its discovery date and some context. Keep all affected records, then display a small preview.

**Inspection question:** Which original values will allow someone to understand and reproduce M02?

**Documentation:** [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame.copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame.index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.index.html) · [pd.set_option()](https://pandas.pydata.org/docs/reference/api/pandas.set_option.html).


In [ ]:
# The issue log will summarise one finding; this separate evidence table must let a reviewer
# locate and inspect every supporting record.

# Keep the absent endpoint, the discovery endpoint and some identifying context. A table of
# names alone would not demonstrate why duration is unavailable.
evidence_columns = ["fire_name", "state", "discovery_date", "contained_date"]

# loc selects the flagged rows and chosen columns. copy gives us a separate table to
# annotate, so evidence labels do not become new source fields.
evidence_m02 = raw.loc[missing_containment, evidence_columns].copy()

# Exports below omit the DataFrame index. Saving the original label as a normal column
# preserves the route back to raw.loc[source_row]; resetting labels first would lose that
# reference.
evidence_m02["source_row"] = evidence_m02.index

# Repeat M02 in every evidence row so each row can be linked to the same single issue. Many
# evidence rows support one issue, rather than becoming many issue entries.
evidence_m02["issue_id"] = "M02"

# Place references before the original values for easy tracing. This column selection changes
# the arrangement, not the number of supporting records.
evidence_m02 = evidence_m02[[
    "issue_id", "source_row", "fire_name", "state", "discovery_date", "contained_date"
]]
pd.set_option("display.max_columns", None)
print("Rows in full M02 evidence:", len(evidence_m02))
# Keep the full table in memory and show ten rows only to make inspection manageable. The
# reported evidence length should agree with the full missing-containment count.
display(evidence_m02.head(10))


### 6.3 Record the measured facts

The affected population contains missing dates. The checked population contains every source record, because each record can be checked for absence.

**Inspection question:** What did we find, and how many records did we actually check?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict) · [int()](https://docs.python.org/3.14/library/functions.html#int) · [round()](https://docs.python.org/3.14/library/functions.html#round).


In [ ]:
# Separate measured facts from written judgement so the count can be checked independently of
# our interpretation.

# A dictionary keeps named answers for ONE issue. Converting it to a table later gives one
# summary row even when thousands of source rows support that finding.
log_entry = {

    # Use M02 consistently in the log and evidence so the summary points to the correct
    # supporting records.
    "issue_id": "M02",

    # Describe what the check establishes. An absence or a disagreement is observable;
    # calling a source value an error would require additional evidence.
    "finding_type": 'Observed absence',

    "field": 'contained_date',

    # missing_containment_count counts the full flagged population. int stores a plain
    # whole-number count; the length of a display preview is not an alternative count.
    "affected_rows": int(missing_containment_count),

    # total_records is the population that could actually be tested. Excluding untestable
    # records from this denominator prevents us from making the issue appear smaller.
    "checked_rows": int(total_records),

    # Keep four decimal places because rounding a very small non-zero result to two could
    # display 0.00%. An unavailable percentage remains unavailable; it is not evidence of a
    # pass.
    "percent": round(missing_containment_count / total_records * 100, 4),
}


### 6.4 Connect the facts to the analysis and evidence

The counts describe the observation. We now explain why we checked it and where a reviewer can find the supporting records.

**Inspection question:** Could another student reproduce this finding and locate its evidence?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict).


In [ ]:
# Add the reason for the check and a route back to its evidence; counts alone do not explain
# why the finding matters.

# The question determines relevance. Missing dates matter to duration; a date-year
# disagreement matters to annual grouping. A different question may lead to a different
# priority.
log_entry["question"] = 'Calculate discovery-to-containment duration for all records.'

# State the condition that would make the question answerable. This connects an acceptance or
# comparison rule to its purpose rather than treating it as an arbitrary instruction.
log_entry["expectation"] = 'A duration needs both a discovery date and a containment date.'

# A notebook section and a reproducible condition let another person rerun the check. The
# issue label by itself would not explain how we arrived at the count.
log_entry["check_reference"] = 'Notebook 01, section 6.1: raw["contained_date"].isna()'

# This answer stores the name evidence_m02 as text; the actual DataFrame remains a separate
# object. The name tells the reviewer which table to consult, but does not save that table.
log_entry["evidence_table"] = "evidence_m02"

# Say whether every affected row was retained. Without that distinction, a ten-row example
# could be mistaken for the full evidence population.
log_entry["evidence_scope"] = "All affected rows; head(10) displays a preview only"


### 6.5 Explain the effect and propose the next step

We have evidence, but inspection alone does not settle how the data should be treated. The following wording is a worked judgement for the stated question.

**Inspection question:** What could this finding change in our analysis, and what should happen next?


In [ ]:
# Write a justified decision proposal: the detection code supplies observations, while these
# answers explain their analytical consequences.

# Connect the observation to a result we might report. For example, a record without a
# containment endpoint cannot supply duration, even when its size can still be analysed.
log_entry["impact"] = 'Duration cannot be calculated for these records from the available dates.'

# Priority follows the stated question, not a percentage cut-off. The explanation below must
# make the chosen priority defensible for this particular use.
log_entry["severity"] = 'High'

# This is worked wording for the supplied reference finding. Compare it with your computed
# result and analysis population before reusing it.
log_entry["severity_reason"] = (
    'Nearly half the records cannot support the stated duration question; the usable '
    'subset may differ.'
)

# Keep the decision as text at the inspection stage. Investigating recovery, defining a
# subset or agreeing a comparison policy is a proposal; assigning this answer changes no
# source values.
log_entry["proposed_action"] = (
    'Check whether dates can be recovered. Otherwise agree a documented duration '
    'subset and report its limits; retain records for other questions.'
)

# Evidence exists, but the treatment has not been agreed or verified. Keeping the issue open
# prevents a proposed action being read as a completed correction.
log_entry["status"] = "Open - decision required"

# Name the later checks that could show the decision worked. Repeating the relevant check and
# reconciling counts is more convincing than merely writing "fixed".
log_entry["verification"] = (
    'Recheck available dates and date order; report usable and excluded counts before '
    'duration analysis.'
)


### 6.6 Display, verify and optionally save the entry

The completed dictionary will become one row in `issue_log`. Keep that summary with the full `evidence_m02` evidence table. Each notebook creates its own one-entry log; they are not combined automatically. To add it to your shared log, match fields by meaning and update an existing M02 entry instead of duplicating it.

**Inspection question:** Does the recorded affected count agree with the evidence we kept?

**Documentation:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [pd.set_option()](https://pandas.pydata.org/docs/reference/api/pandas.set_option.html) · [DataFrame.T](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.T.html) · [DataFrame.to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len).


In [ ]:
# Show the one-issue summary and compare its affected count with the complete supporting
# evidence before exporting.

# The outer list contains one dictionary, so Pandas interprets its keys as headings and its
# answers as one row. Re-running recreates this table; it does not append a second issue.
issue_log = pd.DataFrame([log_entry])

# Long written answers need to stay visible for review. Transposing the displayed table puts
# field names down the page; the stored issue_log still has one issue row.
pd.set_option("display.max_colwidth", None)
display(issue_log.T)

# These two counts should agree because one evidence row represents one flagged source
# record. A difference would suggest the evidence filter or the logged count came from a
# different check.
print("Rows in full evidence:", len(evidence_m02))
print("Affected rows recorded in log:", log_entry["affected_rows"])

# OPTIONAL: uncomment the next two commands to save both complete tables. index=False avoids
# a redundant index column; source_row still preserves traceability. Saving again replaces
# these CSVs, so retain reusable wording in the notebook.
# issue_log.to_csv("Issue_Log_M02.csv", index=False)
# evidence_m02.to_csv("evidence_m02.csv", index=False)


## 7. Findings and the next investigation

| Check | What to carry forward |
| --- | --- |
| S01: structure | All eight expected fields are present; compare types with their meanings. |
| M01: missing names | 960,479 records; determine which questions need names. |
| M02: missing containment | 891,531 records; a worked issue entry and full evidence table are above. |
| Missingness overlap | 651,288 records lack both name and containment date; counts overlap. |
| R01: exact repetition | 365,548 group members; source identity still needs investigation. |

Do not add these counts to produce a “total errors” figure. The same record can appear in several checks.
Continue with Notebook 02 to inspect the values themselves.


### Study check

Explain why `missing_counts` has field names as its index while `missing_names` has source-row labels.
Then explain why the missing-name and missing-date counts cannot simply be added.

**Practice:** Create M01 for missing names using section 4.1. State a question that needs names, keep the relevant
evidence, and justify its severity. Use one issue-log row for the finding, not one row per affected record.

**Next:** Notebook 02 checks whether populated cells contain useful labels, whether formatting splits name groups,
and whether numbers and dates can be interpreted.
